# Dual-stage breast ultrasound: reproduce the paper on Kaggle

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100) · Settings → **Internet ON** · *Add Input* → your private dataset (the folder that contains `manifest.csv`).

Run the sections in order. Each section can also be run in a later session: grids skip finished runs, and the `best_run` cells only read results from `/kaggle/working/runs`. To continue in a new session, attach the previous notebook output as input and run the **Resume** cell.

## 0. Setup

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <-- change this
BRANCH = "main"

# Private repo? Add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets) and uncomment:
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret("GITHUB_TOKEN")
# REPO_URL = REPO_URL.replace("https://", f"https://{token}@")

!rm -rf /kaggle/working/repo
!git clone -q --depth 1 -b {BRANCH} {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements-kaggle.txt
!pip install -q -e . --no-deps

In [ ]:
import glob, os, sys
sys.path.insert(0, "/kaggle/working/repo/src")

hits = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
assert hits, "Dataset not found: add the dataset that contains manifest.csv via 'Add Input'"
DATA = os.path.dirname(hits[0])
os.environ["BUS_DATA_ROOT"] = DATA
RUNS = "/kaggle/working/runs"
REPORTS = "/kaggle/working/reports"
print("data:", DATA)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Resume (only in a new session): copy runs from a previous notebook output attached as input
# !cp -r /kaggle/input/YOUR-PREVIOUS-OUTPUT/runs /kaggle/working/

## 1. Check the code (≈ 3 min)
Unit tests, then a *smoke* run of every component on a handful of images for 1 epoch.

In [ ]:
!python -m pytest -q tests

In [ ]:
S = "/kaggle/working/smoke"
!python -m bus.train_seg --config configs/seg/base.yaml --out {S} --smoke
!python -m bus.train_cls --config configs/cls/base.yaml --out {S} --smoke
!python -m bus.evaluate --seg {S}/seg/A/deeplabv3plus_resnet34_c64/seed42 --cls {S}/cls/A/mobilenetv3_small_mlp_k10_d0.5/seed42 --scenario A --out {S} --smoke
!python -m bus.baselines.maskrcnn --config configs/baselines/maskrcnn.yaml --out {S} --smoke
!python -m bus.baselines.multitask_swin --config configs/baselines/multitask_swin.yaml --out {S} --smoke
!rm -rf {S}

## 2. Table 1: segmentation grid, Scenario A (13 runs)

In [ ]:
!python -m bus.grid configs/grids/table1_seg.yaml --out {RUNS}

In [ ]:
from bus.reporting import best_run
SEG_A = best_run(f"{RUNS}/seg/A", "segmentation.dice")   # chosen on VALIDATION Dice

## 3. Table 2: classification grid, Scenario A (56 runs; ViT-B is the slowest)
Use `--only mobilenet efficientnet` to run a subset first.

In [ ]:
!python -m bus.grid configs/grids/table2_cls.yaml --out {RUNS}

In [ ]:
CLS_A = best_run(f"{RUNS}/cls/A", "classification.auc")   # chosen on VALIDATION AUC

## 4. Full pipeline: Scenario A, Scenario C, and external BUS-BRA

In [ ]:
!python -m bus.evaluate --seg {SEG_A} --cls {CLS_A} --scenario A --role test --out {RUNS}
!python -m bus.evaluate --seg {SEG_A} --cls {CLS_A} --scenario C --role test --out {RUNS}
!python -m bus.evaluate --seg {SEG_A} --cls {CLS_A} --scenario A --role external --out {RUNS}

## 5. Scenario B: retrain the best configurations on BUSI + Breast-Lesions-USG

In [ ]:
!python -m bus.train_seg --config {SEG_A}/config.yaml --set data.scenario=B --out {RUNS}
!python -m bus.train_cls --config {CLS_A}/config.yaml --set data.scenario=B --out {RUNS}
SEG_B = best_run(f"{RUNS}/seg/B", "segmentation.dice")
CLS_B = best_run(f"{RUNS}/cls/B", "classification.auc")
!python -m bus.evaluate --seg {SEG_B} --cls {CLS_B} --scenario B --role test --out {RUNS}

## 6. Baselines: Mask R-CNN and multi-task Swin (Scenarios A, B, C)

In [ ]:
!python -m bus.baselines.maskrcnn --config configs/baselines/maskrcnn.yaml --out {RUNS}
!python -m bus.baselines.maskrcnn --config configs/baselines/maskrcnn.yaml --set data.scenario=B --out {RUNS}
MRC_A = best_run(f"{RUNS}/maskrcnn/A", "segmentation.dice")
!python -m bus.baselines.maskrcnn --config configs/baselines/maskrcnn.yaml --ckpt {MRC_A} --set data.scenario=C "eval.roles=[test]" --out {RUNS}

In [ ]:
!python -m bus.baselines.multitask_swin --config configs/baselines/multitask_swin.yaml --out {RUNS}
!python -m bus.baselines.multitask_swin --config configs/baselines/multitask_swin.yaml --set data.scenario=B --out {RUNS}
SWIN_A = best_run(f"{RUNS}/mtswin/A", "segmentation.dice")
!python -m bus.baselines.multitask_swin --config configs/baselines/multitask_swin.yaml --ckpt {SWIN_A} --set data.scenario=C "eval.roles=[test]" --out {RUNS}

## 7. Statistics and result tables

In [ ]:
OURS_A = f"{RUNS}/pipeline/A/dualstage/per_image_test.csv"
!python -m bus.stats --a {OURS_A} --b {MRC_A}/per_image_test.csv --names ours maskrcnn --out {RUNS}/stats_A_ours_vs_maskrcnn.json
!python -m bus.stats --a {OURS_A} --b {SWIN_A}/per_image_test.csv --names ours mtswin --out {RUNS}/stats_A_ours_vs_mtswin.json

In [ ]:
!python -m bus.reporting collect {RUNS} --out {REPORTS}
from IPython.display import Markdown, display
display(Markdown(open(f"{REPORTS}/results.md").read()))

## 8. Package results for download
`results.zip` holds metrics, per-image CSVs, figures and logs (no checkpoints). `best_models.zip` holds the checkpoints of the chosen models.

In [ ]:
%cd /kaggle/working
!zip -qr results.zip runs reports -x "*.pt" "*/pred_masks/*"
!zip -qr best_models.zip {SEG_A}/best.pt {SEG_A}/config.yaml {CLS_A}/best.pt {CLS_A}/config.yaml
!ls -lh results.zip best_models.zip